In [ ]:
from IPython.core.interactiveshell import InteractiveShell
%load_ext autoreload
InteractiveShell.ast_node_interactivity = "all"
import logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)

In [ ]:
import numpy as np
import pandas as pd
import os
import sys
import matplotlib.pyplot as plt

module_path = os.path.abspath(
    os.path.join("/cmnfs/proj/ORIGINS/protMSD/maxquant/ScanByScan/swaps")
)
if module_path not in sys.path:
    sys.path.append(module_path)

# MQ MBR result

In [ ]:
evidence_MBR = pd.read_csv(
    "/cmnfs/proj/ORIGINS/data/HeLa_sample_amount_and_LC_columns/30min_5ug_ref/txt_30min_5ug_MBR/evidence.txt",
    sep="\t",
)
evidence_MBR["Raw file"].unique()
evidence_MBR = evidence_MBR.loc[evidence_MBR["Reverse"] != "+", :]

In [134]:
MBR_R1 = evidence_MBR[evidence_MBR["Raw file"] == "Hela_30min_5ug_R1_RA1_1_5162"]
MBR_R2 = evidence_MBR[evidence_MBR["Raw file"] == "Hela_30min_5ug_R2_RA1_1_5163"]
MBR_R3 = evidence_MBR[evidence_MBR["Raw file"] == "Hela_30min_5ug_R3_RA1_1_5164"]

In [159]:
def sum_intensity(df):
    df = df.groupby(
        ["Mod. peptide ID", "Charge", "Type", "Raw file"], as_index=False
    ).agg({"Intensity": ["sum", "count"]})
    df.columns = [
        "Mod. peptide ID",
        "Charge",
        "Type",
        "Raw file",
        "Intensity_sum",
        "Intensity_count",
    ]
    return df

In [160]:
evidence_MBR_sum = sum_intensity(evidence_MBR)

In [136]:
MBR_R1_sum = sum_intensity(MBR_R1)
MBR_R2_sum = sum_intensity(MBR_R2)
MBR_R3_sum = sum_intensity(MBR_R3)

In [ ]:
MBR_R1_sum["Intensity_count"].value_counts()

In [138]:
MQ_R1_R2 = pd.merge(
    MBR_R1_sum,
    MBR_R2_sum,
    on=["Mod. peptide ID", "Charge"],
    suffixes=("_R1", "_R2"),
)
MQ_R2_R3 = pd.merge(MBR_R2_sum, MBR_R3_sum, on=["Mod. peptide ID", "Charge"])
MQ_R2_R3 = pd.merge(
    MBR_R1_sum,
    MBR_R3_sum,
    on=["Mod. peptide ID", "Charge"],
    suffixes=("_R1", "_R3"),
)

## Identification count point

In [ ]:
import matplotlib.pyplot as plt

plt.rc("font", size=24)  # Set the default font size for all text elements
plt.figure(figsize=(3, 6))  # Set the size of the figure
evidence_MBR_sum["count"] = 1
evidence_MBR_sum_pivot = evidence_MBR_sum.pivot_table(
    index="Raw file",
    columns="Type",
    values="count",
    aggfunc="sum",
    fill_value=0,
)
evidence_MBR_sum_pivot = evidence_MBR_sum_pivot[["TIMS-MULTI-MSMS", "TIMS-MULTI-MATCH"]]
ax = evidence_MBR_sum_pivot.plot(
    kind="bar",
    stacked=True,
    legend=True,
    # hue_order=["TIMS-MULTI-MSMS", "TIMS-MULTI-MATCH"],
)
ax.legend(bbox_to_anchor=(1.01, 1))
for idx_col, col in enumerate(evidence_MBR_sum_pivot.columns):
    for idx, val in enumerate(evidence_MBR_sum_pivot.index):
        ax.text(
            x=idx,
            y=(idx_col + 1) * 27000,
            s=f"{evidence_MBR_sum_pivot.loc[val, col]}",
            ha="center",
            va="bottom",
        )
for spine in ax.spines.values():
    spine.set_linewidth(3)  # Increase this value for thicker lines
plt.xticks(rotation=0)
plt.xticks([0, 1, 2], ["R1", "R2", "R3"])
plt.xlabel("Experiment")
# plt.savefig(
#     "/cmnfs/proj/ORIGINS/SWAPS_exp/SWAPS_paper_figures/revision/comparison_MBR/swap_mq_id.png",
#     bbox_inches="tight",
#     dpi=300,
# )

In [ ]:
58457 / 64828
59587 / 64828
59471 / 64828

## Run-to-run pair-wise correlation

In [ ]:
# All ID results
from utils.plot import plot_scatter

reg_int, abs_residue, valid_idx = plot_scatter(
    x=MQ_R1_R2["Intensity_sum_R1"],
    y=MQ_R1_R2["Intensity_sum_R2"],
    title="",
    x_label="MaxQuant Intensity R1 (Log10)",
    y_label="MaxQuant Intensity R2 (Log10)",
    log_x=True,
    log_y=True,
    font_size=20,
    save_dir="/cmnfs/proj/ORIGINS/SWAPS_exp/SWAPS_paper_figures/revision/comparison_MBR/",
    fig_spec_name="scatter_MQ_all_R1_R2",
)

In [ ]:
# MSMSdd ID results
from utils.plot import plot_scatter

MQ_R1_R2_MSMS = MQ_R1_R2.loc[
    (MQ_R1_R2["Type_R1"] == "TIMS-MULTI-MSMS")
    & (MQ_R1_R2["Type_R2"] == "TIMS-MULTI-MSMS")
]
reg_int, abs_residue, valid_idx = plot_scatter(
    x=MQ_R1_R2_MSMS["Intensity_sum_R1"],
    y=MQ_R1_R2_MSMS["Intensity_sum_R2"],
    title="",
    x_label="MaxQuant Intensity R1 (Log10)",
    y_label="MaxQuant Intensity R2 (Log10)",
    log_x=True,
    log_y=True,
    font_size=20,
    save_dir="/cmnfs/proj/ORIGINS/SWAPS_exp/SWAPS_paper_figures/revision/comparison_MBR/",
    fig_spec_name="scatter_MQ_msms_R1_R2",
)

In [ ]:
# Match ID results
from utils.plot import plot_scatter

MQ_R1_R2_MATCH = MQ_R1_R2.loc[
    (MQ_R1_R2["Type_R1"] != "TIMS-MULTI-MSMS")
    | (MQ_R1_R2["Type_R2"] != "TIMS-MULTI-MSMS")
]

reg_int, abs_residue, valid_idx = plot_scatter(
    x=MQ_R1_R2_MATCH["Intensity_sum_R1"],
    y=MQ_R1_R2_MATCH["Intensity_sum_R2"],
    title="",
    x_label="MaxQuant Intensity R1 (Log10)",
    y_label="MaxQuant Intensity R2 (Log10)",
    log_x=True,
    log_y=True,
    font_size=20,
    save_dir="/cmnfs/proj/ORIGINS/SWAPS_exp/SWAPS_paper_figures/revision/comparison_MBR/",
    fig_spec_name="scatter_MQ_match_R1_R2",
)

In [ ]:
# Match ID results
MQ_R1_R2_MATCH_filtered = MQ_R1_R2_MATCH.loc[
    (MQ_R1_R2_MATCH["Intensity_count_R1"] == 1)
    & (MQ_R1_R2_MATCH["Intensity_count_R2"] == 1)
]
reg_int, abs_residue, valid_idx = plot_scatter(
    x=MQ_R1_R2_MATCH_filtered["Intensity_sum_R1"],
    y=MQ_R1_R2_MATCH_filtered["Intensity_sum_R2"],
    title="",
    x_label="MaxQuant Intensity R1 (Log10)",
    y_label="MaxQuant Intensity R2 (Log10)",
    log_x=True,
    log_y=True,
    font_size=20,
    save_dir="/cmnfs/proj/ORIGINS/SWAPS_exp/SWAPS_paper_figures/revision/comparison_MBR/",
    fig_spec_name="scatter_MQ_match_filter_multi_elution_R1_R2",
)

# SWAPS MBR

In [3]:
def read_swaps_result(result_path, spec_dir: str = "", read_ps: bool = True):
    maxquant_result_ref = pd.read_pickle(
        os.path.join(result_path, "maxquant_result_ref.pkl")
    )
    if read_ps:
        ps_act = pd.read_csv(
            os.path.join(result_path, "peak_selection", spec_dir, "pept_act_sum_ps.csv")
        )
    else:
        ps_act = pd.read_csv(
            os.path.join(
                result_path, "results", "activation", "pept_act_sum_filter_by_im.csv"
            )
        )
    ps_act_full_df = pd.merge(
        left=ps_act, right=maxquant_result_ref, on=["mz_rank"], how="left"
    )
    return ps_act_full_df

In [14]:
pept_act_sum_col = [
    "Sequence",
    "Modifications",
    "Modified sequence",
    "Charge",
    "sum_intensity",
    "source",
]
merge_exp_on_col = ["Sequence", "Modifications", "Modified sequence", "Charge"]

## With Peak Selection

In [5]:
result_path = "/cmnfs/proj/ORIGINS/SWAPS_exp/short_gradient/30min_3to45_7R_30min_exp_library_no_decoy_MBR_R2_20250113_135150_122755"
R2 = read_swaps_result(result_path, spec_dir="exp_20250114_130627_202099", read_ps=True)
result_path = "/cmnfs/proj/ORIGINS/SWAPS_exp/short_gradient/30min_3to45_7R_30min_exp_library_no_decoy_MBR_R3_20250113_141233_668739"
R3 = read_swaps_result(result_path, spec_dir="exp_20250114_130326_353987", read_ps=True)
result_path = "/cmnfs/proj/ORIGINS/SWAPS_exp/short_gradient/30min_3to45_7R_30min_exp_library_no_decoy_MBR_R1_20250113_141233_681815"
R1 = read_swaps_result(result_path, spec_dir="exp_20250114_091248_079583", read_ps=True)
R1.rename(columns={"pept_act_sum_filter_by_im": "sum_intensity"}, inplace=True)
R2.rename(columns={"pept_act_sum_filter_by_im": "sum_intensity"}, inplace=True)
R3.rename(columns={"pept_act_sum_filter_by_im": "sum_intensity"}, inplace=True)
fig_dir = (
    "/cmnfs/proj/ORIGINS/SWAPS_exp/SWAPS_paper_figures/revision/comparison_MBR/with_PS/"
)

In [6]:
R1_R2 = pd.merge(
    R1[pept_act_sum_col],
    R2[pept_act_sum_col],
    on=merge_exp_on_col,
    suffixes=("_R1", "_R2"),
)
R2_R3 = pd.merge(
    R2[pept_act_sum_col],
    R3[pept_act_sum_col],
    on=merge_exp_on_col,
    suffixes=("_R2", "_R3"),
)
R1_R3 = pd.merge(
    R1[pept_act_sum_col],
    R3[pept_act_sum_col],
    on=merge_exp_on_col,
    suffixes=("_R1", "_R3"),
)

### Identification count plot

In [7]:
int_thres = 100
R1_to_combine = R1.loc[
    R1["sum_intensity"] > int_thres,
    pept_act_sum_col,
]
R1_to_combine["Experiment"] = "R1"
R2_to_combine = R2.loc[
    R2["sum_intensity"] > int_thres,
    pept_act_sum_col,
]
R2_to_combine["Experiment"] = "R2"
R3_to_combine = R3.loc[
    R3["sum_intensity"] > int_thres,
    pept_act_sum_col,
]
R3_to_combine["Experiment"] = "R3"
swaps_all = pd.concat([R1_to_combine, R2_to_combine, R3_to_combine]).reset_index(
    drop=True
)

In [ ]:
import matplotlib.pyplot as plt

swaps_all.loc[swaps_all["source"] == "both", "Source"] = "MQ ID"
swaps_all.loc[swaps_all["source"] == "ref", "Source"] = "SWA Gain"
plt.rc("font", size=24)  # Set the default font size for all text elements
plt.figure(figsize=(3, 6))  # Set the size of the figure
swaps_all["count"] = 1
swaps_all_pivot = swaps_all.pivot_table(
    index="Experiment",
    columns="Source",
    values="count",
    aggfunc="sum",
    fill_value=0,
)
ax = swaps_all_pivot.plot(kind="bar", stacked=True, legend=True)
ax.legend(bbox_to_anchor=(1.01, 1))
for idx_col, col in enumerate(swaps_all_pivot.columns):
    for idx, val in enumerate(swaps_all_pivot.index):
        ax.text(
            x=idx,
            y=(idx_col * 2 + 0.75) * 20000,
            s=f"{swaps_all_pivot.loc[val, col]}",
            ha="center",
            va="bottom",
        )
for spine in ax.spines.values():
    spine.set_linewidth(3)  # Increase this value for thicker lines
plt.xticks(rotation=0)
plt.savefig(
    os.path.join(fig_dir, "swap_mq_id.png"),
    bbox_inches="tight",
    dpi=300,
)

### Run-to-run pair-wise Correlation

In [ ]:
R1_R2[["sum_intensity_R1", "sum_intensity_R2"]].apply(lambda x: np.log10(x + 1)).corr(
    method="pearson"
)
R2_R3[["sum_intensity_R2", "sum_intensity_R3"]].apply(lambda x: np.log10(x + 1)).corr(
    method="pearson"
)
R1_R3[["sum_intensity_R1", "sum_intensity_R3"]].apply(lambda x: np.log10(x + 1)).corr(
    method="pearson"
)

In [ ]:
# All ID results
from utils.plot import plot_scatter

reg_int, abs_residue, valid_idx = plot_scatter(
    x=R1_R2["sum_intensity_R1"],
    y=R1_R2["sum_intensity_R2"],
    title="",
    x_label="SWA Inferred Intensity R1 (Log10)",
    y_label="SWA Inferred Intensity R2 (Log10)",
    log_x=True,
    log_y=True,
    font_size=20,
    save_dir=fig_dir,
    fig_spec_name="scatter_swaps_all_R1_R2",
)

In [ ]:
# Only Exp
from utils.plot import plot_scatter

R1_R2_exp = R1_R2.loc[(R1_R2["source_R1"] == "both") & (R1_R2["source_R2"] == "both")]
reg_int, abs_residue, valid_idx = plot_scatter(
    x=R1_R2_exp["sum_intensity_R1"],
    y=R1_R2_exp["sum_intensity_R2"],
    title="",
    x_label="SWA Inferred Intensity R1 (Log10)",
    y_label="SWA Inferred Intensity R2 (Log10)",
    log_x=True,
    log_y=True,
    font_size=20,
    save_dir=fig_dir,
    fig_spec_name="scatter_swaps_exp_R1_R2",
)

In [ ]:
# Only Ref
from utils.plot import plot_scatter

R1_R2_ref = R1_R2.loc[(R1_R2["source_R1"] != "both") | (R1_R2["source_R2"] != "both")]
reg_int, abs_residue, valid_idx = plot_scatter(
    x=R1_R2_ref["sum_intensity_R1"],
    y=R1_R2_ref["sum_intensity_R2"],
    title="",
    x_label="SWA Inferred Intensity R1 (Log10)",
    y_label="SWA Inferred Intensity R2 (Log10)",
    log_x=True,
    log_y=True,
    font_size=20,
    save_dir=fig_dir,
    fig_spec_name="scatter_swaps_ref_R1_R2",
)

## Without Peak Selection

In [15]:
result_path = "/cmnfs/proj/ORIGINS/SWAPS_exp/short_gradient/30min_3to45_7R_30min_exp_library_no_decoy_MBR_R2_20250113_135150_122755"
R2 = read_swaps_result(result_path, read_ps=False)
result_path = "/cmnfs/proj/ORIGINS/SWAPS_exp/short_gradient/30min_3to45_7R_30min_exp_library_no_decoy_MBR_R3_20250113_141233_668739"
R3 = read_swaps_result(result_path, read_ps=False)
result_path = "/cmnfs/proj/ORIGINS/SWAPS_exp/short_gradient/30min_3to45_7R_30min_exp_library_no_decoy_MBR_R1_20250113_141233_681815"
R1 = read_swaps_result(result_path, read_ps=False)
R1.rename(columns={"pept_act_sum_filter_by_im": "sum_intensity"}, inplace=True)
R2.rename(columns={"pept_act_sum_filter_by_im": "sum_intensity"}, inplace=True)
R3.rename(columns={"pept_act_sum_filter_by_im": "sum_intensity"}, inplace=True)
fig_dir = (
    "/cmnfs/proj/ORIGINS/SWAPS_exp/SWAPS_paper_figures/revision/comparison_MBR/no_PS/"
)

In [ ]:
R1_R2 = pd.merge(
    R1[pept_act_sum_col],
    R2[pept_act_sum_col],
    on=merge_exp_on_col,
    suffixes=("_R1", "_R2"),
)
R2_R3 = pd.merge(
    R2[pept_act_sum_col],
    R3[pept_act_sum_col],
    on=merge_exp_on_col,
    suffixes=("_R2", "_R3"),
)
R2_R3 = pd.merge(
    R1[pept_act_sum_col],
    R3[pept_act_sum_col],
    on=merge_exp_on_col,
    suffixes=("_R1", "_R3"),
)

### Identification count plot

In [18]:
int_thres = 100
R1_to_combine = R1.loc[R1["sum_intensity"] > int_thres, pept_act_sum_col]
R1_to_combine["Experiment"] = "R1"
R2_to_combine = R2.loc[R2["sum_intensity"] > int_thres, pept_act_sum_col]
R2_to_combine["Experiment"] = "R2"
R3_to_combine = R3.loc[R3["sum_intensity"] > int_thres, pept_act_sum_col]
R3_to_combine["Experiment"] = "R3"
swaps_all = pd.concat([R1_to_combine, R2_to_combine, R3_to_combine]).reset_index(
    drop=True
)

In [ ]:
import matplotlib.pyplot as plt

swaps_all.loc[swaps_all["source"] == "both", "Source"] = "MQ ID"
swaps_all.loc[swaps_all["source"] == "ref", "Source"] = "SWA Gain"
plt.rc("font", size=24)  # Set the default font size for all text elements
plt.figure(figsize=(3, 6))  # Set the size of the figure
swaps_all["count"] = 1
swaps_all_pivot = swaps_all.pivot_table(
    index="Experiment",
    columns="Source",
    values="count",
    aggfunc="sum",
    fill_value=0,
)
ax = swaps_all_pivot.plot(kind="bar", stacked=True, legend=True)
ax.legend(bbox_to_anchor=(1.01, 1))
for idx_col, col in enumerate(swaps_all_pivot.columns):
    for idx, val in enumerate(swaps_all_pivot.index):
        ax.text(
            x=idx,
            y=(idx_col * 2 + 1) * 2000,
            s=f"{swaps_all_pivot.loc[val, col]}",
            ha="center",
            va="bottom",
        )
for spine in ax.spines.values():
    spine.set_linewidth(3)  # Increase this value for thicker lines
plt.xticks(rotation=0)
plt.savefig(
    os.path.join(fig_dir, "swap_mq_id.png"),
    bbox_inches="tight",
    dpi=300,
)

### Run-to-run pair-wise Correlation

In [ ]:
R1_R2[["sum_intensity_R1", "sum_intensity_R2"]].apply(lambda x: np.log10(x + 1)).corr(
    method="pearson"
)
R2_R3[["sum_intensity_R2", "sum_intensity_R3"]].apply(lambda x: np.log10(x + 1)).corr(
    method="pearson"
)
R1_R3[["sum_intensity_R1", "sum_intensity_R3"]].apply(lambda x: np.log10(x + 1)).corr(
    method="pearson"
)

#### R1 vs R2

In [ ]:
# All ID results
from utils.plot import plot_scatter

reg_int, abs_residue, valid_idx = plot_scatter(
    x=R1_R2["sum_intensity_R1"],
    y=R1_R2["sum_intensity_R2"],
    title="",
    x_label="SWA Inferred Intensity R1 (Log10)",
    y_label="SWA Inferred Intensity R2 (Log10)",
    log_x=True,
    log_y=True,
    font_size=20,
    save_dir=fig_dir,
    fig_spec_name="scatter_swaps_all_R1_R2",
)

In [ ]:
# Only Exp
from utils.plot import plot_scatter

R1_R2_exp = R1_R2.loc[(R1_R2["source_R1"] == "both") & (R1_R2["source_R2"] == "both")]
reg_int, abs_residue, valid_idx = plot_scatter(
    x=R1_R2_exp["sum_intensity_R1"],
    y=R1_R2_exp["sum_intensity_R2"],
    title="",
    x_label="SWA Inferred Intensity R1 (Log10)",
    y_label="SWA Inferred Intensity R2 (Log10)",
    log_x=True,
    log_y=True,
    font_size=20,
    save_dir=fig_dir,
    fig_spec_name="scatter_swaps_exp_R1_R2",
)

In [ ]:
# Only Ref
from utils.plot import plot_scatter

R1_R2_ref = R1_R2.loc[(R1_R2["source_R1"] != "both") | (R1_R2["source_R2"] != "both")]
reg_int, abs_residue, valid_idx = plot_scatter(
    x=R1_R2_ref["sum_intensity_R1"],
    y=R1_R2_ref["sum_intensity_R2"],
    title="",
    x_label="SWA Inferred Intensity R1 (Log10)",
    y_label="SWA Inferred Intensity R2 (Log10)",
    log_x=True,
    log_y=True,
    font_size=20,
    save_dir=fig_dir,
    fig_spec_name="scatter_swaps_ref_R1_R2",
)

# What is discovered by SWAPS but not by MaxQuant